# Extract completed Lean scripts from Nemotron Math Proofs v1

This notebook reads the 25+ GB source JSONL one record at a time, so memory use does not grow with the dataset size. It writes one JSON object per completed assistant response to a new JSONL file.

In [1]:
import json
import re
from itertools import islice
from pathlib import Path
from typing import Any


REPO_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
INPUT_PATH = REPO_ROOT / "data" / "dataset" / "nemotron_math_proofs_v1.jsonl"
OUTPUT_PATH = REPO_ROOT / "data" / "dataset" / "nemotron_math_proofs_v1_finished_lean_proofs.jsonl"
PROGRESS_EVERY = 100_000

LEAN_CODE_BLOCK = re.compile(r"```lean4[ \t]*\n(?P<code>.*)\n?```", re.DOTALL)
UNFINISHED_PROOF = re.compile(r"\b(?:sorry|admit)\b")

## Dataset structure

Each physical line is an independent JSON object with these fields:

- `problem`: the original natural-language problem.
- `source`: the source dataset, such as `aops` or `mathstack`.
- `formal_statement`: the Lean declarations and target theorem. The target intentionally ends in `by sorry`; this is the prompt, not the completed proof.
- `lean_header`: imports, options, and `open` commands used by the formal statement.
- `url`, `user_name`, `user_url`, `license`: source attribution.
- `sft_line_number`, `used_in`, `tools`: dataset bookkeeping.
- `uuid`: the record identifier.
- `split`: `lean` for these formal proof records.
- `messages`: either empty or a `user`/`assistant` conversation. The assistant's `content` is a complete script inside one `lean4` Markdown code block; `reasoning_content` contains the model's hidden proof reasoning.

The extractor takes only the assistant's final fenced script, removes the Markdown fence, and rejects scripts that still contain `sorry` or `admit`. The output records contain only `lean_proof`. This identifies completed model responses; it does not re-run Lean to prove that every generated script compiles.

In [2]:
with INPUT_PATH.open(encoding="utf-8") as input_file:
    sample_records = [json.loads(line) for line in islice(input_file, 10)]

first_record = sample_records[0]
first_completed_record = next(record for record in sample_records if record["messages"])
assistant_message = first_completed_record["messages"][-1]

structure_summary = {
    "input_size_gb": round(INPUT_PATH.stat().st_size / 1_000_000_000, 2),
    "top_level_fields": {key: type(value).__name__ for key, value in first_record.items()},
    "message_roles": [message["role"] for message in first_completed_record["messages"]],
    "assistant_fields": list(assistant_message),
    "formal_statement_ending": first_completed_record["formal_statement"][-80:],
    "assistant_content_ending": assistant_message["content"][-80:],
}
structure_summary

{'input_size_gb': 29.59,
 'top_level_fields': {'problem': 'str',
  'source': 'str',
  'formal_statement': 'str',
  'lean_header': 'str',
  'url': 'NoneType',
  'user_name': 'NoneType',
  'user_url': 'NoneType',
  'sft_line_number': 'NoneType',
  'messages': 'list',
  'uuid': 'str',
  'used_in': 'list',
  'tools': 'list',
  'license': 'str',
  'split': 'str'},
 'message_roles': ['user', 'assistant'],
 'assistant_fields': ['role', 'content', 'reasoning_content'],
 'formal_statement_ending': 'ype*} [TopologicalSpace α] (x : α) :\n    MyIsConnected ({x} : Set α) := by sorry',
 'assistant_content_ending': '({x} : Set α) := by\n    exact ⟨h_nonempty, h_preconnected⟩\n  \n  exact h_main\n```'}

In [3]:
def extract_lean_script(record: dict[str, Any]) -> str | None:
    if record["split"] != "lean" or not record["messages"]:
        return None

    assistant_message = record["messages"][-1]
    if assistant_message["role"] != "assistant":
        return None

    match = LEAN_CODE_BLOCK.fullmatch(assistant_message["content"].strip())
    if match is None:
        return None

    script = match.group("code").strip()
    if UNFINISHED_PROOF.search(script):
        return None

    return script


def extract_finished_proofs(input_path: Path, output_path: Path) -> dict[str, int]:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    records_seen = 0
    proofs_written = 0

    with (
        input_path.open(encoding="utf-8") as input_file,
        temporary_path.open("w", encoding="utf-8") as output_file,
    ):
        for records_seen, line in enumerate(input_file, start=1):
            record = json.loads(line)
            lean_script = extract_lean_script(record)
            if lean_script is not None:
                output_record = {"lean_proof": lean_script}
                output_file.write(json.dumps(output_record, ensure_ascii=False) + "\n")
                proofs_written += 1

            if records_seen % PROGRESS_EVERY == 0:
                print(f"Read {records_seen:,} records; wrote {proofs_written:,} proofs")

    temporary_path.replace(output_path)
    return {"records_seen": records_seen, "proofs_written": proofs_written}

## Run the extraction

This scans the entire source file and can take a while. Data is first written to a `.tmp` file and moved to the final path only after a successful complete scan.

In [4]:
extraction_stats = extract_finished_proofs(INPUT_PATH, OUTPUT_PATH)
extraction_stats

Read 100,000 records; wrote 65,871 proofs
Read 200,000 records; wrote 132,199 proofs
Read 300,000 records; wrote 198,600 proofs
Read 400,000 records; wrote 264,849 proofs
Read 500,000 records; wrote 331,453 proofs
Read 600,000 records; wrote 398,054 proofs
Read 700,000 records; wrote 464,360 proofs
Read 800,000 records; wrote 530,748 proofs
Read 900,000 records; wrote 596,755 proofs
Read 1,000,000 records; wrote 662,950 proofs
Read 1,100,000 records; wrote 729,750 proofs
Read 1,200,000 records; wrote 796,691 proofs
Read 1,300,000 records; wrote 863,373 proofs


{'records_seen': 1376663, 'proofs_written': 914521}

In [5]:
with OUTPUT_PATH.open(encoding="utf-8") as output_file:
    output_samples = [json.loads(line) for line in islice(output_file, 2)]

validation_summary = {
    "output_path": str(OUTPUT_PATH),
    "output_size_gb": round(OUTPUT_PATH.stat().st_size / 1_000_000_000, 2),
    "sample_proof_lengths": [len(record["lean_proof"]) for record in output_samples],
    "sample_proof_starts": [record["lean_proof"][:160] for record in output_samples],
}
validation_summary

{'output_path': '/work3/s204164/mini-alphaproof/data/dataset/nemotron_math_proofs_v1_finished_lean_proofs.jsonl',
 'output_size_gb': 5.66,
 'sample_proof_lengths': [2037, 2188],
 'sample_proof_starts': ['import Mathlib\nimport Aesop\nimport Mathlib.Topology.Basic\nset_option maxHeartbeats 0\nopen BigOperators Real Nat Topology Rat\nopen Set\n/-- Is a single point topo',
  'import Mathlib\nimport Aesop\nimport Mathlib.Topology.Basic\nset_option maxHeartbeats 0\nopen BigOperators Real Nat Topology Rat\nopen Set\n/-- Is a single point topo']}